In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
items_schema = ArrayType(
    StructType(
        [
            StructField("item_id", StringType()),
            StructField("name", StringType()),
            StructField("category", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DecimalType(10,2)),
            StructField("subtotal", DecimalType(10,2))
        ]
    )
)

df_fact_orders = spark.table("01_bronze.orders")\
                .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))\
                .withColumn("order_date", to_date(col("order_timestamp")))\
                .withColumn("items_parsed", from_json(col("items"), items_schema))\
                .withColumn("item", explode(col("items_parsed")))\
                    .select(
            "order_id",
            col("item.item_id").alias("item_id"),
            "restaurant_id",
            "order_timestamp",
            "order_date",
            col("item.name").alias("item_name"),
            col("item.category").alias("category"),
            col("item.quantity").alias("quantity"),
            col("item.unit_price").cast("decimal(10,2)").alias("unit_price"),
            col("item.subtotal").cast("decimal(10,2)").alias("subtotal"),
        )
    
display(df_fact_orders)

order_id,item_id,restaurant_id,order_timestamp,order_date,item_name,category,quantity,unit_price,subtotal
ORD-20250715-104297,ITEM-205,REST-AUH-001,2025-07-15T13:59:33.174Z,2025-07-15,Chole Bhature,Main Course,1,40.45,40.45
ORD-20250715-104297,ITEM-502,REST-AUH-001,2025-07-15T13:59:33.174Z,2025-07-15,Rasmalai (2 pcs),Dessert,1,18.41,18.41
ORD-20250715-104297,ITEM-604,REST-AUH-001,2025-07-15T13:59:33.174Z,2025-07-15,Fresh Lime Soda,Beverage,1,11.77,11.77
ORD-20250715-104297,ITEM-501,REST-AUH-001,2025-07-15T13:59:33.174Z,2025-07-15,Gulab Jamun (2 pcs),Dessert,2,15.28,30.56
ORD-20250715-104297,ITEM-401,REST-AUH-001,2025-07-15T13:59:33.174Z,2025-07-15,Naan,Bread,2,7.82,15.64
ORD-20250715-108293,ITEM-302,REST-DXB-001,2025-07-15T11:02:16.174Z,2025-07-15,Chicken Tikka Masala,Main Course,2,49.85,99.70
ORD-20250715-108293,ITEM-601,REST-DXB-001,2025-07-15T11:02:16.174Z,2025-07-15,Masala Chai,Beverage,2,12.30,24.60
ORD-20250715-108293,ITEM-301,REST-DXB-001,2025-07-15T11:02:16.174Z,2025-07-15,Butter Chicken,Main Course,2,52.06,104.12
ORD-20250715-108293,ITEM-202,REST-DXB-001,2025-07-15T11:02:16.174Z,2025-07-15,Dal Makhani,Main Course,1,36.22,36.22
ORD-20250715-108293,ITEM-604,REST-DXB-001,2025-07-15T11:02:16.174Z,2025-07-15,Fresh Lime Soda,Beverage,3,11.57,34.71
